# Feature Engineering and Modelling

---

1. Import packages
2. Load data
3. Modelling

---

## 1. Import packages

In [49]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [50]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

# Shows plots in jupyter notebook
%matplotlib inline

# Set plot style
sns.set(color_codes=True)

---
## 2. Load data

In [51]:
df = pd.read_csv(r"C:\Users\KIRONJI\Desktop\BCG Data science simulation\data\clean\data_for_predictions.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,...,months_modif_prod,months_renewal,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_usilxuppasemubllopkaafesmlibmsdf,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw
0,24011ae4ebbe3035111d65fa7c15bc57,0.000000,4.739944,0.000000,0.000000,0.0,0.444045,0.114481,0.098142,40.606701,...,2,6,0,0,1,0,0,0,0,1
1,d29c2c54acc38ff3c0614d0a653813dd,3.668479,0.000000,0.000000,2.280920,0.0,1.237292,0.145711,0.000000,44.311378,...,76,4,1,0,0,0,0,1,0,0
2,764c75f661154dac3a6c254cd082ea7d,2.736397,0.000000,0.000000,1.689841,0.0,1.599009,0.165794,0.087899,44.311378,...,68,8,0,0,1,0,0,1,0,0
3,bba03439a292a1e166f80264c16191cb,3.200029,0.000000,0.000000,2.382089,0.0,1.318689,0.146694,0.000000,44.311378,...,69,9,0,0,0,1,0,1,0,0
4,149d57cf92fc41cf94415803a877cb4b,3.646011,0.000000,2.721811,2.650065,0.0,2.122969,0.116900,0.100015,40.606701,...,71,9,1,0,0,0,0,1,0,0


---

## 3. Modelling

We now have a dataset containing features that we have engineered and we are ready to start training a predictive model. Remember, we only need to focus on training a `Random Forest` classifier.

In [52]:
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

### Data sampling

The first thing we want to do is split our dataset into training and test samples. The reason why we do this, is so that we can simulate a real life situation by generating predictions for our test sample, without showing the predictive model these data points. This gives us the ability to see how well our model is able to generalise to new data, which is critical.

A typical % to dedicate to testing is between 20-30, for this example we will use a 75-25% split between train and test respectively.

In [53]:
# Make a copy of our data
train_df = df.copy()

# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [54]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42, stratify=y)
print(y_train.mean(), y_test.mean())
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

0.09713346722658389 0.09720700985761227
(10954, 61)
(10954,)
(3652, 61)
(3652,)


* Added the stratify parameter since the churners make up around 9.7% of the population-an even split would will produce different churn rates in the train and test split

### Model training

Once again, we are using a `Random Forest` classifier in this example. A Random Forest sits within the category of `ensemble` algorithms because internally the `Forest` refers to a collection of `Decision Trees` which are tree-based learning algorithms. As the data scientist, you can control how large the forest is (that is, how many decision trees you want to include).

The reason why an `ensemble` algorithm is powerful is because of the laws of averaging, weak learners and the central limit theorem. If we take a single decision tree and give it a sample of data and some parameters, it will learn patterns from the data. It may be overfit or it may be underfit, but that is now our only hope, that single algorithm. 

With `ensemble` methods, instead of banking on 1 single trained model, we can train 1000's of decision trees, all using different splits of the data and learning different patterns. It would be like asking 1000 people to all learn how to code. You would end up with 1000 people with different answers, methods and styles! The weak learner notion applies here too, it has been found that if you train your learners not to overfit, but to learn weak patterns within the data and you have a lot of these weak learners, together they come together to form a highly predictive pool of knowledge! This is a real life application of many brains are better than 1.

Now instead of relying on 1 single decision tree for prediction, the random forest puts it to the overall views of the entire collection of decision trees. Some ensemble algorithms using a voting approach to decide which prediction is best, others using averaging. 

As we increase the number of learners, the idea is that the random forest's performance should converge to its best possible solution.

Some additional advantages of the random forest classifier include:

- The random forest uses a rule-based approach instead of a distance calculation and so features do not need to be scaled
- It is able to handle non-linear parameters better than linear based models

On the flip side, some disadvantages of the random forest classifier include:

- The computational power needed to train a random forest on a large dataset is high, since we need to build a whole ensemble of estimators.
- Training time can be longer due to the increased complexity and size of thee ensemble

In [55]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6, 8, 10, 15],
    'min_samples_leaf': [5, 10, 20, 40],
    'max_features': ['sqrt', 'log2', 0.3],
    'class_weight': ['balanced', 'balanced_subsample'],
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    scoring='average_precision',
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    random_state=42,
    refit=True,
)
search.fit(X_train, y_train)
print(search.best_params_, search.best_score_)
best = search.best_estimator_

{'n_estimators': 500, 'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': 15, 'class_weight': 'balanced'} 0.27645495584385493


# Explanation for the parameters:
- More trees means the random forest will follow the central limit theorem thus converging to the optimal prediction
- We need fairly small tree depths since an unlimited tree just memorizes the training data
- A high minimum sample leaf avoids overfitting-less leaves means the data 
- class_weighting- since we have significantly less churners(90/10), the model ought to be penalised more for missing a churner
- max_features- Ensures each splits only sees a subset of data generally improving the ensemble feature
- random state-ensures a consistent outcome throughout 

**Drawing from all this cocnlusions,I came up with ranges for each of the parameters and Tuned them with a randomized search CV- i don't make guesses but rather chose the optimal values**

**The average_precision scoring rewards ranking churners above non churners thus a better metric than accuracy**

**The startified CV ensures every candidate is scored on data it did not train on**

- This yields the following results for parameters;'n_estimators': 500, 'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': 15, 'class_weight': 'balanced

*I introduce a dummy classifier-one that flags churners based of the more frequent label to use it as a baseline for evaluation*

 *Added default random forest classifier as well to see how it differs from the rules based dummy classifier*

In [56]:
from sklearn.dummy import DummyClassifier

def report(name, model, thr=0.5):
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= thr).astype(int)
    print(f"{name} | thr={thr:.2f} | acc={metrics.accuracy_score(y_test, pred):.3f} "
          f"prec={metrics.precision_score(y_test, pred, zero_division=0):.3f} "
          f"rec={metrics.recall_score(y_test, pred):.3f} "
          f"f1={metrics.f1_score(y_test, pred):.3f} "
          f"roc_auc={metrics.roc_auc_score(y_test, p):.3f} "
          f"pr_auc={metrics.average_precision_score(y_test, p):.3f}")
    return p

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
report('dummy', dummy)

base = RandomForestClassifier(random_state=42).fit(X_train, y_train)
report('default RF', base)
print('default RF train ROC-AUC:', metrics.roc_auc_score(y_train, base.predict_proba(X_train)[:, 1]))

dummy | thr=0.50 | acc=0.903 prec=0.000 rec=0.000 f1=0.000 roc_auc=0.500 pr_auc=0.097
default RF | thr=0.50 | acc=0.907 prec=0.718 rec=0.079 f1=0.142 roc_auc=0.692 pr_auc=0.289
default RF train ROC-AUC: 1.0


**The key evaluation metric of choice is recall--alligning with powerco's goal of customer retention, it is more costly to miss churner than wasting a retention offer on a customer who wasn't leaving, thus the best model is one that flags more churners**

**PR-AUC is another metric used through out-to assess how well the model maintains precision while achieveing different levels of recall(looking below, PR AUC has a steady increase from 9.7% on a dummy model to 28.9% on a default RF and to 30.6% for the tuned RF- the model; It looks weak on a 0-1 scale but relative to the baseilnes we can see the model improves)**


- The dummy gets 90% accuracy while flagging nobody.
- The default forest’s 90.7% accuracy is barely better than the dummy’s. It flags under 8% of churners.
- Its train ROC-AUC is 1.000- It memorised the training data, which is  the overfitting that max_depth and min_samples_leaf are there to fix.

In [57]:
p = report('tuned RF', best)
print('train ROC-AUC:', metrics.roc_auc_score(y_train, best.predict_proba(X_train)[:, 1]))
print(metrics.confusion_matrix(y_test, (p >= 0.5).astype(int)))
print(metrics.classification_report(y_test, (p >= 0.5).astype(int)))

tuned RF | thr=0.50 | acc=0.861 prec=0.304 rec=0.332 f1=0.318 roc_auc=0.704 pr_auc=0.306
train ROC-AUC: 0.9981141237826618
[[3027  270]
 [ 237  118]]
              precision    recall  f1-score   support

           0       0.93      0.92      0.92      3297
           1       0.30      0.33      0.32       355

    accuracy                           0.86      3652
   macro avg       0.62      0.63      0.62      3652
weighted avg       0.87      0.86      0.86      3652



- The tuned random forest has a higher recall rate(33%) compared to the default(7.9%)- the tuned RF, even with a lower precision rate , was able to flag more churners
- PR AUC moves up as well from 0.289 on defualt to 0.306 on the tuned model- a positive sign

In [58]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(
    best, X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method='predict_proba'
)[:, 1]

prec, rec, thr = metrics.precision_recall_curve(y_train, oof)
# Business rule: catch at least 65% of churners
idx = np.where(rec[:-1] >= 0.65)[0][-1]
chosen_thr = thr[idx]
print('chosen threshold:', chosen_thr)

best.fit(X_train, y_train)
p = best.predict_proba(X_test)[:, 1]
final_pred = (p >= chosen_thr).astype(int)
print(metrics.confusion_matrix(y_test, final_pred))
print(metrics.classification_report(y_test, final_pred))

chosen threshold: 0.31516774697339506
[[1951 1346]
 [ 105  250]]
              precision    recall  f1-score   support

           0       0.95      0.59      0.73      3297
           1       0.16      0.70      0.26       355

    accuracy                           0.60      3652
   macro avg       0.55      0.65      0.49      3652
weighted avg       0.87      0.60      0.68      3652



- To find a suitable threshold, I used out of fold predictions with cross_val_predict, compared the predicted values to actual labels from the train dataset.

- Then I chose a threshold that flagged atleast 65% of the churners-used it on the test split

- This achieved a recall of 70%- 70% of all the churners were flagged-Thus the tuned RF with a threshold(~0.315) flags churners more efficiently

- It finds 250 of 355 churners but flags 46% of customers, about 1.6× better than random targeting.

# Is the model satisfactory?

The model is not a confident churn predictor, but usable as a prioritisation tool.

What's good:
- PR-AUC is 0.306 against a random baseline of 0.097, about 3× better than chance.
- ROC-AUC is 0.70, so there's real signal.
- It's stable: CV and test scores are close.

What's weak:

- At the threshold that catches 70% of churners, only 16% of flagged customers actually churn. It flags 44% of all customers to find those churners.
- A 0.70 ROC-AUC is mediocre. It's fine as a first model but not strong.
- Still overfit (0.998 train AUC).

For PowerCo: if a retention offer is cheap compared to losing a customer, 16% precision can still pay off. If offers are expensive, it won't.

Feature importance: the top drivers are the net and gross power margins, then consumption, meter rent and tenure. 

The hypothesis, the Dec–Jan off-peak price gap doesnt hold- price changes aren't a strong standalone churn driver in this model.
